# Финальная модель: точно как лучший Optuna trial + весь test батчами

Модель обучается ровно так же, как внутри `objective`: только на `train_pool`, без `eval_set` и без early stopping. `val_pool` используется после обучения только для расчёта PR-AUC. Затем весь test считается четырьмя батчами и объединяется в один итоговый PR-AUC.

In [1]:
# %%
# Импорты и настройки

import gc
import json
from pathlib import Path

import numpy as np
import polars as pl

from catboost import CatBoostRegressor, Pool
from scipy.special import ndtr
from sklearn.metrics import average_precision_score


HORIZON = 24.0

TO_DROP = [
    "ид_события",
    "ид_канала_данных",
    "дата",
    "время",
    "datetime",
    "hours_since_prev_target",
]

TARGET_COLS = [
    "target_lower",
    "target_upper",
]


# Те же чанки, что были в 4_tuning.ipynb
TRAIN_CHUNKS = [*range(8)]
VAL_CHUNKS = [*range(80)]


# Тот же feature selection step
SELECTED_STEP = 28

HISTORY_PATH = Path(
    "feature_selection_shap_rfe/history.csv"
)

OUTPUT_DIR = Path(
    "survival_optuna"
)

BEST_PARAMS_PATH = (
    OUTPUT_DIR / "best_params.json"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_model.cbm"
)

SCORES_PATH = (
    OUTPUT_DIR / "scores_final.csv"
)

TEST_DIR = Path(
    "data/final_test"
)


N_TEST_BATCHES = 4
RANDOM_STATE = 42

In [2]:
# %%
# Та же функция чтения split, что в тюнинге

def scan_split(
    split: str,
    chunks: list[int],
) -> pl.LazyFrame:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .drop(TO_DROP)
    )


schema = scan_split(
    "train",
    [TRAIN_CHUNKS[0]],
).collect_schema()


feature_cols = [
    c
    for c in schema.names()
    if c not in TARGET_COLS
]


cat_features = [
    c
    for c in feature_cols
    if schema[c] in (
        pl.String,
        pl.Categorical,
        pl.Enum,
    )
]


print(f"All features: {len(feature_cols)}")
print(f"Cat features: {len(cat_features)}")

All features: 378
Cat features: 4


In [3]:
# %%
# Точно так же восстанавливаем фичи выбранного шага

history_df = pl.read_csv(
    HISTORY_PATH
)


removed_rows = (
    history_df
    .filter(
        pl.col("step") <= SELECTED_STEP
    )
    .select("removed_features")
    .drop_nulls()
    ["removed_features"]
    .to_list()
)


removed_features = {
    feature
    for row in removed_rows
    for feature in row.split(" | ")
    if feature
}


selected_features = [
    feature
    for feature in feature_cols
    if feature not in removed_features
]


selected_cat_features = [
    feature
    for feature in cat_features
    if feature in selected_features
]


expected_n_features = (
    history_df
    .filter(
        pl.col("step") == SELECTED_STEP
    )
    ["n_features"]
    .item()
)


assert (
    len(selected_features)
    == expected_n_features
)


print(f"Step: {SELECTED_STEP}")
print(f"Selected features: {len(selected_features)}")
print(f"Expected features: {expected_n_features}")
print(f"Selected cat features: {len(selected_cat_features)}")

Step: 28
Selected features: 98
Expected features: 98
Selected cat features: 2


In [4]:
# %%
# Та же make_pool, что использовалась перед Optuna

def make_pool(
    split: str,
    chunks: list[int],
) -> tuple[Pool, np.ndarray, int]:

    lf = (
        scan_split(
            split,
            chunks,
        )
        .select(
            selected_features
            + TARGET_COLS
        )
    )

    if selected_cat_features:
        lf = lf.with_columns(
            pl.col(
                selected_cat_features
            ).fill_null("null")
        )

    df = lf.collect(
        engine="streaming"
    )

    y_survival = (
        df
        .select(TARGET_COLS)
        .to_numpy()
    )

    y_binary = (
        (
            (df["target_upper"] != -1)
            & (
                df["target_lower"]
                <= HORIZON
            )
        )
        .cast(pl.Int8)
        .to_numpy()
    )

    pool = Pool(
        data=df.select(
            selected_features
        ),
        label=y_survival,
        cat_features=selected_cat_features,
    )

    n_rows = df.height

    del df
    del y_survival

    gc.collect()

    return pool, y_binary, n_rows


train_pool, y_train_binary, n_train = make_pool(
    "train",
    TRAIN_CHUNKS,
)

val_pool, y_val_binary, n_val = make_pool(
    "val",
    VAL_CHUNKS,
)


print(f"Train rows: {n_train:,}")
print(f"Val rows:   {n_val:,}")

Train rows: 9,523,050
Val rows:   6,871,013


In [5]:
# %%
# Та же функция преобразования SurvivalAft prediction -> P(event <= 24h)

def aft_to_probability(
    pred: np.ndarray,
    scale: float,
) -> np.ndarray:

    return ndtr(
        (
            np.log(HORIZON)
            - pred
        )
        / scale
    )

In [6]:
# %%
# Загружаем best_params, найденные Optuna

with open(
    BEST_PARAMS_PATH,
    "r",
    encoding="utf-8",
) as f:
    best = json.load(f)


best_scale = float(
    best.pop("scale")
)


# Точно те же параметры модели, что были внутри objective
best_model_params = {
    "loss_function": (
        f"SurvivalAft:"
        f"dist=Normal;"
        f"scale={best_scale}"
    ),
    **best,
    "random_seed": RANDOM_STATE,
    "verbose": 1,
    "allow_writing_files": False,
}


print(f"Best scale: {best_scale:.6f}")
print(f"Best params: {best_model_params}")

Best scale: 0.651915
Best params: {'loss_function': 'SurvivalAft:dist=Normal;scale=0.6519151388952278', 'iterations': 100, 'depth': 9, 'learning_rate': 0.14762522256363309, 'l2_leaf_reg': 22.647264220860485, 'random_seed': 42, 'verbose': 1, 'allow_writing_files': False}


In [7]:
# %%
# ВАЖНО:
# обучаем РОВНО как в objective()
# validation сюда НЕ передаётся
# early stopping НЕ используется

model = CatBoostRegressor(
    **best_model_params
)


model.fit(
    train_pool
)


model.save_model(
    BEST_MODEL_PATH
)


print()
print(f"Tree count: {model.tree_count_}")
print(f"Model saved to: {BEST_MODEL_PATH}")

0:	learn: 45.7089762	total: 2.2s	remaining: 3m 37s
1:	learn: 44.4966156	total: 3.89s	remaining: 3m 10s
2:	learn: 42.8164651	total: 6s	remaining: 3m 14s
3:	learn: 40.6270574	total: 7.82s	remaining: 3m 7s
4:	learn: 37.9157742	total: 9.9s	remaining: 3m 8s
5:	learn: 34.6973701	total: 11.6s	remaining: 3m 1s
6:	learn: 31.0077739	total: 13.6s	remaining: 3m
7:	learn: 26.8939192	total: 15.2s	remaining: 2m 55s
8:	learn: 22.4095602	total: 16.9s	remaining: 2m 51s
9:	learn: 17.6034946	total: 18.6s	remaining: 2m 47s
10:	learn: 12.5381416	total: 20.5s	remaining: 2m 46s
11:	learn: 7.2569786	total: 22.4s	remaining: 2m 44s
12:	learn: 1.8043197	total: 24.2s	remaining: 2m 41s
13:	learn: 0.5339988	total: 26s	remaining: 2m 39s
14:	learn: 0.5428214	total: 27.6s	remaining: 2m 36s
15:	learn: 0.5842373	total: 29.2s	remaining: 2m 33s
16:	learn: 0.6373314	total: 31s	remaining: 2m 31s
17:	learn: 0.6871709	total: 32.6s	remaining: 2m 28s
18:	learn: 0.7500241	total: 34.2s	remaining: 2m 25s
19:	learn: 0.8020477	total:

In [8]:
# %%
# Validation считаем ТОЧНО как внутри objective()

pred_val = model.predict(
    val_pool
)


p_val = aft_to_probability(
    pred_val,
    best_scale,
)


val_pr_auc = average_precision_score(
    y_val_binary,
    p_val,
)


print(f"VAL PR-AUC: {val_pr_auc:.6f}")


del pred_val
del p_val

gc.collect()

VAL PR-AUC: 0.767664


0

In [9]:
# %%
# Делим ВСЕ test-файлы на 4 батча
# Ничего из test не выбрасывается

test_files = sorted(
    TEST_DIR.glob("chunk_*.parquet")
)


if not test_files:
    raise FileNotFoundError(
        f"No parquet files found in {TEST_DIR}"
    )


test_batches = [
    batch.tolist()
    for batch in np.array_split(
        np.array(
            test_files,
            dtype=object,
        ),
        N_TEST_BATCHES,
    )
]


print(f"Test files: {len(test_files)}")

for i, batch_files in enumerate(
    test_batches,
    start=1,
):
    print(
        f"Batch {i}: "
        f"{len(batch_files)} files"
    )

Test files: 116
Batch 1: 29 files
Batch 2: 29 files
Batch 3: 29 files
Batch 4: 29 files


In [10]:
# %%
# Predict по ВСЕМУ test четырьмя батчами

y_test_parts = []
p_test_parts = []

n_test = 0


for batch_idx, batch_files in enumerate(
    test_batches,
    start=1,
):

    lf = (
        pl.scan_parquet(
            batch_files
        )
        .drop(TO_DROP)
        .select(
            selected_features
            + TARGET_COLS
        )
    )

    if selected_cat_features:
        lf = lf.with_columns(
            pl.col(
                selected_cat_features
            ).fill_null("null")
        )

    df = lf.collect(
        engine="streaming"
    )

    y_chunk = (
        (
            (df["target_upper"] != -1)
            & (
                df["target_lower"]
                <= HORIZON
            )
        )
        .cast(pl.Int8)
        .to_numpy()
    )

    test_pool = Pool(
        data=df.select(
            selected_features
        ),
        cat_features=selected_cat_features,
    )

    pred_chunk = model.predict(
        test_pool
    )

    p_chunk = aft_to_probability(
        pred_chunk,
        best_scale,
    )

    y_test_parts.append(
        y_chunk
    )

    p_test_parts.append(
        p_chunk
    )

    n_test += df.height

    print(
        f"Batch {batch_idx}/{N_TEST_BATCHES}: "
        f"{df.height:,} rows"
    )

    del df
    del test_pool
    del y_chunk
    del pred_chunk
    del p_chunk
    del lf

    gc.collect()

Batch 1/4: 2,334,407 rows
Batch 2/4: 3,420,301 rows
Batch 3/4: 5,792,369 rows
Batch 4/4: 524,960 rows


In [11]:
# %%
# Один итоговый PR-AUC по ВСЕМУ test

y_test = np.concatenate(
    y_test_parts
)

p_test = np.concatenate(
    p_test_parts
)


assert len(y_test) == n_test
assert len(p_test) == n_test


test_pr_auc = average_precision_score(
    y_test,
    p_test,
)


positive_rate = y_test.mean()


result = pl.DataFrame(
    {
        "tree_count": [
            model.tree_count_
        ],
        "val_pr_auc": [
            val_pr_auc
        ],
        "test_rows": [
            n_test
        ],
        "test_positive_rate": [
            positive_rate
        ],
        "test_pr_auc": [
            test_pr_auc
        ],
    }
)


result.write_csv(
    SCORES_PATH
)


print()
print(f"Tree count:     {model.tree_count_}")
print(f"VAL PR-AUC:     {val_pr_auc:.6f}")
print(f"TEST rows:      {n_test:,}")
print(f"Positive rate:  {positive_rate:.6f}")
print(f"TEST PR-AUC:    {test_pr_auc:.6f}")
print(f"Saved to:       {SCORES_PATH}")


result


Tree count:     100
VAL PR-AUC:     0.767664
TEST rows:      12,072,037
Positive rate:  0.014113
TEST PR-AUC:    0.438573
Saved to:       survival_optuna/scores_final.csv


tree_count,val_pr_auc,test_rows,test_positive_rate,test_pr_auc
i64,f64,i64,f64,f64
100,0.767664,12072037,0.014113,0.438573
